In [1]:
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DateType,
    DoubleType,
)

spark = SparkSession.builder.master("local[*]").appName("DataFrameFilteringPractice").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/19 18:33:10 WARN Utils: Your hostname, DESKTOP-440A2SS, resolves to a loopback address: 127.0.1.1; using 172.20.131.26 instead (on interface eth0)
26/10/19 18:33:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/19 18:33:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
data = [
    (
        1,
        "Иван Петров",
        datetime.strptime("2023-09-15", "%Y-%m-%d").date(),
        ("Ноутбук", 1, 999.99),
    ),
    (
        2,
        "Анна Сидорова",
        datetime.strptime("2023-09-16", "%Y-%m-%d").date(),
        ("Смартфон", 2, 599.50),
    ),
    (
        3,
        "Елена Иванова",
        datetime.strptime("2023-09-17", "%Y-%m-%d").date(),
        ("Наушники", 3, 79.99),
    ),
    (
        4,
        "Анна Козлова",
        datetime.strptime("2023-09-18", "%Y-%m-%d").date(),
        ("Смартфон", 1, 699.99),
    ),
    (
        5,
        "Петр Смирнов",
        datetime.strptime("2023-09-19", "%Y-%m-%d").date(),
        ("Планшет", 1, 449.99),
    ),
]

In [3]:
schema = StructType(
    [
        StructField("order_id", IntegerType(), False),
        StructField("customer_name", StringType(), False),
        StructField("order_date", DateType(), False),
        StructField(
            "items",
            StructType(
                [
                    StructField("product_name", StringType(), False),
                    StructField("quantity", IntegerType(), False),
                    StructField("price", DoubleType(), False),
                ]
            ),
            False,
        ),
    ]
)

In [4]:
df = spark.createDataFrame(data, schema)

In [6]:
df.show()

+--------+-------------+----------+--------------------+
|order_id|customer_name|order_date|               items|
+--------+-------------+----------+--------------------+
|       1|  Иван Петров|2023-09-15|{Ноутбук, 1, 999.99}|
|       2|Анна Сидорова|2023-09-16|{Смартфон, 2, 599.5}|
|       3|Елена Иванова|2023-09-17|{Наушники, 3, 79.99}|
|       4| Анна Козлова|2023-09-18|{Смартфон, 1, 699...|
|       5| Петр Смирнов|2023-09-19|{Планшет, 1, 449.99}|
+--------+-------------+----------+--------------------+



In [10]:
import pyspark.sql.functions as F

In [27]:
df_after_15sep = df.filter(F.col("order_date") > "2023-09-15")
df_after_15sep.show()

+--------+-------------+----------+--------------------+
|order_id|customer_name|order_date|               items|
+--------+-------------+----------+--------------------+
|       2|Анна Сидорова|2023-09-16|{Смартфон, 2, 599.5}|
|       3|Елена Иванова|2023-09-17|{Наушники, 3, 79.99}|
|       4| Анна Козлова|2023-09-18|{Смартфон, 1, 699...|
|       5| Петр Смирнов|2023-09-19|{Планшет, 1, 449.99}|
+--------+-------------+----------+--------------------+



In [19]:
df_quantity_gt_1 = df.where(F.col("items.quantity") > 1)
df_quantity_gt_1.show()

+--------+-------------+----------+--------------------+
|order_id|customer_name|order_date|               items|
+--------+-------------+----------+--------------------+
|       2|Анна Сидорова|2023-09-16|{Смартфон, 2, 599.5}|
|       3|Елена Иванова|2023-09-17|{Наушники, 3, 79.99}|
+--------+-------------+----------+--------------------+



In [20]:
df_price_gt_500 = df.where(F.col("items.price") > 500)
df_price_gt_500.show()

+--------+-------------+----------+--------------------+
|order_id|customer_name|order_date|               items|
+--------+-------------+----------+--------------------+
|       1|  Иван Петров|2023-09-15|{Ноутбук, 1, 999.99}|
|       2|Анна Сидорова|2023-09-16|{Смартфон, 2, 599.5}|
|       4| Анна Козлова|2023-09-18|{Смартфон, 1, 699...|
+--------+-------------+----------+--------------------+



In [28]:
df_anna_orders = df.where(F.col("customer_name").startswith("Анна"))
df_anna_orders.show()

+--------+-------------+----------+--------------------+
|order_id|customer_name|order_date|               items|
+--------+-------------+----------+--------------------+
|       2|Анна Сидорова|2023-09-16|{Смартфон, 2, 599.5}|
|       4| Анна Козлова|2023-09-18|{Смартфон, 1, 699...|
+--------+-------------+----------+--------------------+



In [29]:
df_smartphone_orders = df.where(F.col("items.product_name").contains("Смартфон"))
df_smartphone_orders.show(truncate=False)

+--------+-------------+----------+---------------------+
|order_id|customer_name|order_date|items                |
+--------+-------------+----------+---------------------+
|2       |Анна Сидорова|2023-09-16|{Смартфон, 2, 599.5} |
|4       |Анна Козлова |2023-09-18|{Смартфон, 1, 699.99}|
+--------+-------------+----------+---------------------+

